# Exploratory Data Analysis for the Australian Weather Dataset
For this assignment, the exploratory data analysis is split from the rest of the tasks to make the code less connected. This is done after experience gained from the previous assignment that the EDA would be shuffled around the notebook depending on which other aspects of the task needed to be done in which order. This also makes it easier to work on the project in parallel and ensure that once the EDA is finished, it does not have to tested after every change in the main notebook. In summary, the EDA for the Australian Weather Dataset is decoupled from the main notebook to get the advantages of loosely coupled code.



In [7]:
# Import required libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import TargetEncoder

In [8]:
eda_dataset = pd.read_csv('Dataset/weatherAUS.csv')
pd.DataFrame(eda_dataset)
eda_dataset.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


In [9]:
eda_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 145460 entries, 0 to 145459
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Date           145460 non-null  str    
 1   Location       145460 non-null  str    
 2   MinTemp        143975 non-null  float64
 3   MaxTemp        144199 non-null  float64
 4   Rainfall       142199 non-null  float64
 5   Evaporation    82670 non-null   float64
 6   Sunshine       75625 non-null   float64
 7   WindGustDir    135134 non-null  str    
 8   WindGustSpeed  135197 non-null  float64
 9   WindDir9am     134894 non-null  str    
 10  WindDir3pm     141232 non-null  str    
 11  WindSpeed9am   143693 non-null  float64
 12  WindSpeed3pm   142398 non-null  float64
 13  Humidity9am    142806 non-null  float64
 14  Humidity3pm    140953 non-null  float64
 15  Pressure9am    130395 non-null  float64
 16  Pressure3pm    130432 non-null  float64
 17  Cloud9am       89572 non-null   float64


Rows with missing values in the target value are dropped since they offer no value to supervised learning methods.

In [10]:
eda_dataset = eda_dataset.dropna(subset=["RainTomorrow"])

Making a numerical dataset and a categorical dataset to make analysis easier

In [11]:
numerical_dataset = eda_dataset.select_dtypes(include='float64')
categorical_dataset = eda_dataset.select_dtypes(include='str')

Dropping any missing values

In [12]:
# Temporarily removed, might be modified if issues persist in the analysis
# numerical_dataset = numerical_dataset.dropna()
# categorical_dataset = categorical_dataset.dropna()

## Temporary Encoding
For certain parts of the analysis, such as the correlation heatmap, it is necessary to use encoded data. There is therefore a temporary encoding of the categorical and time series values connected to the dataset. The values are encoded in such a way that all parts of features are carried forward in the dataset, and that the encoding makes sense for the feature domain. For example, ordinal encoder is not used on location, since there is no inherent order (other than perhaps coordinate location) to them. Another example is that it may be that the year is unnecessary as a feature since historically rain is hard to predict based on yearly intervals. But, it is still included since this is the analysis part.


In [13]:
# Encoding of Target
categorical_dataset['RainTomorrow'] = categorical_dataset['RainTomorrow'].replace({'No':0, 'Yes':1}).astype(int)

Encoding of categorical data

In [14]:
categorical_dataset.info()

<class 'pandas.DataFrame'>
Index: 142193 entries, 0 to 145458
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   Date          142193 non-null  str  
 1   Location      142193 non-null  str  
 2   WindGustDir   132863 non-null  str  
 3   WindDir9am    132180 non-null  str  
 4   WindDir3pm    138415 non-null  str  
 5   RainToday     140787 non-null  str  
 6   RainTomorrow  142193 non-null  int64
dtypes: int64(1), str(6)
memory usage: 8.7 MB


In [15]:
# Target encoding of location
x = categorical_dataset[['Location']]
y = categorical_dataset['RainTomorrow']

encoder = TargetEncoder()
categorical_dataset['Location'] = encoder.fit_transform(x,y)

In [16]:
# Target encoding of Wind Gust Direction
x = categorical_dataset[['WindGustDir']]
y = categorical_dataset['RainTomorrow']

encoder = TargetEncoder()
categorical_dataset['WindGustDir'] = encoder.fit_transform(x,y)

In [17]:
# Target encoding of Wind Gust Direction at 9 AM
x = categorical_dataset[['WindDir9am']]
y = categorical_dataset['RainTomorrow']

encoder = TargetEncoder()
categorical_dataset['WindDir9am'] = encoder.fit_transform(x,y)

In [18]:
# Target encoding of Wind Gust Direction at 3 PM
x = categorical_dataset[['WindDir3pm']]
y = categorical_dataset['RainTomorrow']

encoder = TargetEncoder()
categorical_dataset['WindDir3pm'] = encoder.fit_transform(x,y)

In [19]:
# Binary Encoding of RainToday
categorical_dataset['RainToday'] = categorical_dataset['RainToday'].replace({'No': 0, 'Yes': 1}).astype(int)

ValueError: cannot convert float NaN to integer

We can see that the encoding was done properly, as the only categorical data left is the Date time series

In [ ]:
categorical_dataset.info()

In [ ]:
numerical_dataset.info()

### Cyclical encoding

This dataset includes multiple features which have cyclical properties. This means that the order of the data is important, but there is a wrap-around of the order at the start and finish. The date includes seasonality, with certain patterns reoccurring depending on which time of year it is. For example, in Australia, rain is more common in the summer. The wind direction is also cyclical, since the direction loops around for every full loop. Ordinal encoding would not retain the wrap-around property that cyclical features contain. Cyclical encoding is used for these features instead.

The relevant features with cyclic properties are as follows:
- WindGustDir
- WindDir9am
- WindDir3am
- date: day
- date: month

A general function for encoding with cyclic data is created which takes a Pandas series object and its corresponding period length. The function returns the same pandas series, encoded for the given period. This encoding includes both a cosine and sine wave fitted version of the Pandas series. Both are needed to keep track of the cycle, since multiple values share the same sine and cosine values in the same cycle. The function only works for numerical data.

In [20]:
def cyclicEncoder(series, period):
    sin = np.sin(2*np.pi*(series/period))
    cos = np.cos(2*np.pi*(series/period))
    return sin, cos

In [23]:
eda_dataset["Date"]

0         2008-12-01
1         2008-12-02
2         2008-12-03
3         2008-12-04
4         2008-12-05
             ...    
145454    2017-06-20
145455    2017-06-21
145456    2017-06-22
145457    2017-06-23
145458    2017-06-24
Name: Date, Length: 142193, dtype: str

In [26]:
# Ready date related data
# Convert to Pandas datetime format
eda_dataset["Date"] = pd.to_datetime(eda_dataset["Date"])
# Extract year, month and day based on year.
eda_dataset["Year"] = eda_dataset["Date"].dt.year
eda_dataset["Month"] = eda_dataset["Date"].dt.month
eda_dataset["DayOfYear"] = eda_dataset["Date"].dt.day_of_year

In [ ]:
# Ready wind related data

## Data Analysis

### Univariate Analysis

In [ ]:
# See how many different locations are used.
eda_dataset["Location"].nunique()

In [ ]:
# Check if every location has the same amount of observations
eda_dataset.value_counts("Location")

As can be seen from the result, the gap between the location with the most and fourth least entries is small, but not negligible. The gap is 3436 - 3001 = 435, which is an increase of about $\frac{435}{3001}$ $\cdot$ 100% $\approx$ 14.5% of the smallest value. This means that the model will be slightly more skewed towards the patterns seen in Canberra and Sydney, compared to the rest of the locations, with three exceptions. The last three locations only have 1578 observations. This is about half of the other 46 locations. These three location will therefore have less of an impact in the model training. It is easier for their patterns to be overshadowed by the other locations which have more instances.


### Multivariate Analysis